In [1]:
# ==============================================================================
# IMPLEMENTASI K-NEAREST NEIGHBOR (K-NN) UNTUK DETEKSI TINGKAT STRES DAN
# KELELAHAN BERBASIS SINYAL FISIOLOGIS TUBUH
#
# Author  : Radithya Farrella Azmi & Shinta Octavia Ramadhani
# Program : S1 Pendidikan Teknik Elektro - Universitas Negeri Malang
# Dataset : Stress-Lysis (Rachakonda et al., 2019) via Kaggle
#
# Eksperimen:
#   1. Baseline K-NN (Z-Score, K=5) -- replikasi laporan
#   2. Perbandingan Normalisasi: RAW vs Z-Score vs Min-Max
#   3. Optimasi Nilai K (K=1 s.d. K=15, selang 2)
#   4. 10-Fold Stratified Cross-Validation (validasi generalisasi)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

# Warna konsisten untuk semua plot
COLOR_RAW    = '#6C757D'   # Abu
COLOR_ZSCORE = '#198754'   # Hijau
COLOR_MINMAX = '#0D6EFD'   # Biru

LABEL_NAMES  = ['Normal', 'Stres', 'Lelah']

# ==============================================================================
# BAGIAN 1 -- LOAD & PREPROCESSING DATASET
# ==============================================================================
print("=" * 60)
print("  BAGIAN 1: LOAD & PREPROCESSING")
print("=" * 60)

try:
    df = pd.read_csv('Stress-Lysis.csv')
    print(f"[OK] Dataset berhasil dimuat: {df.shape[0]} baris, {df.shape[1]} kolom")
except FileNotFoundError:
    print("[ERROR] File 'Stress-Lysis.csv' tidak ditemukan. Upload dulu ke Colab!")
    exit()

# --- Konversi & Mapping ---
# Temperature (Fahrenheit) -> Body Temperature (Celcius)
df['Body_Temp_C']          = (df['Temperature'] - 32) * 5 / 9

# Humidity -> Galvanic Skin Response (GSR) proxy
df['GSR_Conductivity_uS']  = df['Humidity']

# Heart rate -> Heart Rate BPM
df['Heart_Rate_BPM']       = df['Heart rate']

# Stress Level -> Label Kelas
df['Label_Kelas']          = df['Stress Level']

df_final = df[['Heart_Rate_BPM', 'GSR_Conductivity_uS', 'Body_Temp_C', 'Label_Kelas']].copy()

print("\n[Preview] 5 Data Teratas Setelah Konversi:")
print(df_final.head().to_string(index=False))

print("\n[Distribusi Kelas]")
dist = df_final['Label_Kelas'].value_counts().sort_index()
for idx, count in dist.items():
    nama = LABEL_NAMES[idx]
    print(f"  Kelas {idx} ({nama:6s}): {count} sampel ({count/len(df_final)*100:.1f}%)")

# ==============================================================================
# BAGIAN 2 -- BASELINE K-NN (Replikasi Laporan Asli)
# ==============================================================================
print("\n" + "=" * 60)
print("  BAGIAN 2: BASELINE K-NN (Z-Score, K=5, Hold-Out 80:20)")
print("=" * 60)

X = df_final[['Heart_Rate_BPM', 'GSR_Conductivity_uS', 'Body_Temp_C']].values
y = df_final['Label_Kelas'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalisasi Z-Score -- fit HANYA pada training, transform keduanya
scaler_z = StandardScaler()
X_train_z = scaler_z.fit_transform(X_train)
X_test_z  = scaler_z.transform(X_test)        # TIDAK fit ulang, cegah data leakage

knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train_z, y_train)
y_pred_base = knn_baseline.predict(X_test_z)

acc_base = accuracy_score(y_test, y_pred_base)
print(f"\nAkurasi Baseline (Z-Score, K=5): {acc_base*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_base, target_names=LABEL_NAMES))

# Plot Confusion Matrix Baseline
fig, ax = plt.subplots(figsize=(6, 5))
cm_base = confusion_matrix(y_test, y_pred_base)
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Greens',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
ax.set_title('Confusion Matrix – Baseline (Z-Score, K=5)', fontsize=13, fontweight='bold')
ax.set_ylabel('Aktual')
ax.set_xlabel('Prediksi')
plt.tight_layout()
plt.savefig('cm_baseline.png', dpi=150)
plt.show()
print("[Saved] cm_baseline.png")

# ==============================================================================
# BAGIAN 3 -- PERBANDINGAN NORMALISASI: RAW vs Z-SCORE vs MIN-MAX
# ==============================================================================
print("\n" + "=" * 60)
print("  BAGIAN 3: PERBANDINGAN TEKNIK NORMALISASI (K=5)")
print("=" * 60)

# Min-Max -- fit HANYA pada training
scaler_mm = MinMaxScaler()
X_train_mm = scaler_mm.fit_transform(X_train)
X_test_mm  = scaler_mm.transform(X_test)

# RAW (tanpa normalisasi) -- sebagai baseline pembanding tambahan
experiments = {
    'Tanpa Normalisasi (RAW)': (X_train,    X_test),
    'Z-Score (StandardScaler)': (X_train_z,  X_test_z),
    'Min-Max (MinMaxScaler)':   (X_train_mm, X_test_mm),
}

results_norm = {}
for nama, (Xtr, Xte) in experiments.items():
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(Xtr, y_train)
    pred = knn.predict(Xte)
    acc  = accuracy_score(y_test, pred)
    results_norm[nama] = acc
    print(f"  {nama:35s}: Akurasi = {acc*100:.2f}%")

# Plot Bar Perbandingan Normalisasi
fig, ax = plt.subplots(figsize=(8, 5))
colors = [COLOR_RAW, COLOR_ZSCORE, COLOR_MINMAX]
bars = ax.bar(results_norm.keys(), [v*100 for v in results_norm.values()],
              color=colors, edgecolor='white', linewidth=1.5, width=0.5)
ax.set_ylim(95, 101)
ax.set_ylabel('Akurasi (%)', fontsize=12)
ax.set_title('Perbandingan Akurasi berdasarkan Teknik Normalisasi (K=5)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, results_norm.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', labelsize=10)
plt.tight_layout()
plt.savefig('bar_normalisasi.png', dpi=150)
plt.show()
print("[Saved] bar_normalisasi.png")

# ==============================================================================
# BAGIAN 4 -- OPTIMASI NILAI K
# ==============================================================================
print("\n" + "=" * 60)
print("  BAGIAN 4: OPTIMASI NILAI K (K=1 hingga K=15)")
print("=" * 60)

k_range   = list(range(1, 16, 2))   # 1, 3, 5, 7, 9, 11, 13, 15
acc_k_raw = []
acc_k_z   = []
acc_k_mm  = []

print(f"\n{'K':>4} | {'RAW':>10} | {'Z-Score':>10} | {'Min-Max':>10}")
print("-" * 42)

for k in k_range:
    a_r = accuracy_score(y_test,
          KNeighborsClassifier(n_neighbors=k).fit(X_train,    y_train).predict(X_test))
    a_z = accuracy_score(y_test,
          KNeighborsClassifier(n_neighbors=k).fit(X_train_z,  y_train).predict(X_test_z))
    a_m = accuracy_score(y_test,
          KNeighborsClassifier(n_neighbors=k).fit(X_train_mm, y_train).predict(X_test_mm))
    acc_k_raw.append(a_r)
    acc_k_z.append(a_z)
    acc_k_mm.append(a_m)
    print(f"{k:>4} | {a_r*100:>9.2f}% | {a_z*100:>9.2f}% | {a_m*100:>9.2f}%")

# Plot Kurva Akurasi vs K
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_range, [v*100 for v in acc_k_raw], 'o--', color=COLOR_RAW,
        linewidth=2, markersize=7, label='Tanpa Normalisasi (RAW)')
ax.plot(k_range, [v*100 for v in acc_k_z],   's-',  color=COLOR_ZSCORE,
        linewidth=2, markersize=7, label='Z-Score')
ax.plot(k_range, [v*100 for v in acc_k_mm],  '^-',  color=COLOR_MINMAX,
        linewidth=2, markersize=7, label='Min-Max')
ax.axvline(x=5, color='red', linestyle=':', linewidth=1.5, alpha=0.7, label='K=5 (dipilih)')
ax.set_xlabel('Nilai K', fontsize=12)
ax.set_ylabel('Akurasi (%)', fontsize=12)
ax.set_title('Pengaruh Nilai K terhadap Akurasi Klasifikasi K-NN',
             fontsize=13, fontweight='bold')
ax.set_xticks(k_range)
ax.set_ylim(98.5, 101)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('kurva_k_optimasi.png', dpi=150)
plt.show()
print("[Saved] kurva_k_optimasi.png")

# Identifikasi K optimal
best_k_z  = k_range[np.argmax(acc_k_z)]
best_k_mm = k_range[np.argmax(acc_k_mm)]
print(f"\n[Hasil] K optimal Z-Score : K={best_k_z} ({max(acc_k_z)*100:.2f}%)")
print(f"[Hasil] K optimal Min-Max : K={best_k_mm} ({max(acc_k_mm)*100:.2f}%)")

# ==============================================================================
# BAGIAN 5 -- 10-FOLD STRATIFIED CROSS-VALIDATION
# ==============================================================================
print("\n" + "=" * 60)
print("  BAGIAN 5: 10-FOLD STRATIFIED CROSS-VALIDATION (K=5)")
print("=" * 60)

kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Normalisasi ulang menggunakan seluruh dataset untuk CV
X_all_z  = StandardScaler().fit_transform(X)
X_all_mm = MinMaxScaler().fit_transform(X)

cv_raw = cross_val_score(KNeighborsClassifier(n_neighbors=5), X,        y, cv=kf, scoring='accuracy')
cv_z   = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_all_z,  y, cv=kf, scoring='accuracy')
cv_mm  = cross_val_score(KNeighborsClassifier(n_neighbors=5), X_all_mm, y, cv=kf, scoring='accuracy')

print(f"\n{'Fold':>6} | {'RAW':>10} | {'Z-Score':>10} | {'Min-Max':>10}")
print("-" * 46)
for i, (r, z, m) in enumerate(zip(cv_raw, cv_z, cv_mm), 1):
    print(f"{i:>6} | {r*100:>9.2f}% | {z*100:>9.2f}% | {m*100:>9.2f}%")
print("-" * 46)
print(f"{'Mean':>6} | {cv_raw.mean()*100:>9.2f}% | {cv_z.mean()*100:>9.2f}% | {cv_mm.mean()*100:>9.2f}%")
print(f"{'Std':>6} | {cv_raw.std()*100:>9.4f}% | {cv_z.std()*100:>9.4f}% | {cv_mm.std()*100:>9.4f}%")

print("\n[Interpretasi]")
print(f"  Z-Score  -> Mean={cv_z.mean()*100:.2f}%, Std={cv_z.std()*100:.4f}%")
print(f"  Min-Max  -> Mean={cv_mm.mean()*100:.2f}%, Std={cv_mm.std()*100:.4f}%")
print(f"  RAW      -> Mean={cv_raw.mean()*100:.2f}%, Std={cv_raw.std()*100:.4f}%")
print("\n  Z-Score dan Min-Max menghasilkan stabilitas lebih tinggi (std lebih rendah)")
print("  dibandingkan RAW, membuktikan normalisasi meningkatkan konsistensi model.")

# Plot Boxplot CV
fig, ax = plt.subplots(figsize=(8, 5))
bp_data   = [cv_raw*100, cv_z*100, cv_mm*100]
bp_labels = ['RAW\n(Tanpa Normalisasi)', 'Z-Score\n(StandardScaler)', 'Min-Max\n(MinMaxScaler)']
bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], [COLOR_RAW, COLOR_ZSCORE, COLOR_MINMAX]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('Akurasi (%)', fontsize=12)
ax.set_title('Distribusi Akurasi 10-Fold CV per Teknik Normalisasi (K=5)',
             fontsize=13, fontweight='bold')
ax.set_ylim(97, 101.5)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('boxplot_cv.png', dpi=150)
plt.show()
print("[Saved] boxplot_cv.png")

# ==============================================================================
# BAGIAN 6 -- RINGKASAN AKHIR & VALIDASI SAMPEL MANUAL
# ==============================================================================
print("\n" + "=" * 60)
print("  BAGIAN 6: RINGKASAN HASIL EKSPERIMEN")
print("=" * 60)

print("""
+----------------------------------+----------+----------+---------+
| Eksperimen                       | Z-Score  | Min-Max  |   RAW   |
+----------------------------------+----------+----------+---------+
| Akurasi Hold-Out (K=5)           | 100.00%  | 100.00%  | 100.00% |
| CV Mean Accuracy (10-Fold, K=5)  |  99.90%  |  99.90%  |  99.80% |
| CV Std Deviation                 |  0.0020  |  0.0020  |  0.0040 |
| K Optimal (Hold-Out)             |    1-7   |    1-7   |   1-7   |
+----------------------------------+----------+----------+---------+
""")

print("[Kesimpulan]")
print("  1. Dataset StressLysis memiliki separabilitas yang tinggi sehingga")
print("     semua teknik normalisasi mencapai akurasi hold-out 100%.")
print("  2. Namun, normalisasi (Z-Score & Min-Max) terbukti meningkatkan")
print("     STABILITAS model: std CV turun dari 0.40% menjadi 0.20%.")
print("  3. Z-Score dipilih sebagai teknik utama karena lebih robust terhadap")
print("     outlier dibandingkan Min-Max, sesuai standar preprocessing medis.")
print("  4. K=5 terbukti valid: berada di rentang optimal (K=1 hingga K=7).")

# Validasi manual beberapa sampel
print("\n[Validasi Sampel Manual -- 5 Data Acak dari Test Set]")
print(f"{'No':>3} | {'HR':>5} | {'GSR':>6} | {'Temp(C)':>8} | {'Aktual':>10} | {'Prediksi':>10} | Status")
print("-" * 72)
np.random.seed(7)
idx_samples = np.random.choice(len(X_test), 5, replace=False)
for i, idx in enumerate(idx_samples, 1):
    hr   = X_test[idx][0]
    gsr  = X_test[idx][1]
    tmp  = X_test[idx][2]
    act  = LABEL_NAMES[y_test[idx]]
    pred_label = LABEL_NAMES[knn_baseline.predict(X_test_z[idx].reshape(1,-1))[0]]
    status = "BENAR" if act == pred_label else "SALAH"
    print(f"{i:>3} | {hr:>5.0f} | {gsr:>6.2f} | {tmp:>8.2f} | {act:>10} | {pred_label:>10} | {status}")

print("\n" + "=" * 60)
print("  SELESAI -- Semua visualisasi telah disimpan sebagai PNG.")
print("=" * 60)

  BAGIAN 1: LOAD & PREPROCESSING
[ERROR] File 'Stress-Lysis.csv' tidak ditemukan. Upload dulu ke Colab!


NameError: name 'df' is not defined